In [9]:
# Parameters
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '' # '', _detected', '_detected_manual'
seed = 42
query_ratio = 0.2

In [10]:
import torch
import numpy as np
import joblib
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict

In [11]:

# Path to new file
data_path = f"saved_models/{megadescriptor_version}/data{detection}.npz"
encoder_path = f"saved_models/{megadescriptor_version}/label_encoder{detection}.pkl"

# Load everything at once
data = np.load(data_path)

embeddings = data["embeddings"]      # shape (N, D)
labels = data["label_ids"]           # integer labels
# original_labels = data["labels"]     # string labels (optional)

print("Embeddings shape:", embeddings.shape)

# Optional: load encoder if you want inverse_transform
encoder = joblib.load(encoder_path)
names = encoder.inverse_transform(labels)

encoder = joblib.load(encoder_path)
id_to_name = dict(enumerate(encoder.classes_))
name_to_id = {v: k for k, v in id_to_name.items()}


Embeddings shape: (319, 768)


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.2, random_state=seed, stratify=labels
)

In [13]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# Triplet Loss


In [15]:
margin = 0.85

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [86]:
# Add this cell after the existing cells, e.g., after cell defining device

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class TripletDataset(Dataset):
    def __init__(self, embeddings, labels, num_triplets_per_class=10, difficulty_threshold=0.0):
        self.embeddings = torch.tensor(embeddings, dtype=torch.float32)
        self.labels = labels
        self.num_triplets_per_class = num_triplets_per_class
        self.difficulty_threshold = difficulty_threshold
        self.class_indices = defaultdict(list)
        for i, label in enumerate(labels):
            self.class_indices[label].append(i)
        self.triplets = self._generate_triplets()

    def _generate_triplets(self):
        triplets = []
        embeddings = self.embeddings
        max_dist = torch.max(torch.cdist(embeddings, embeddings))  # max distance in dataset
        for class_id, indices in self.class_indices.items():
            if len(indices) < 2:
                continue
            for _ in range(self.num_triplets_per_class * 5):  # sample extra, filter by difficulty
                anchor_idx = np.random.choice(indices)
                positive_idx = np.random.choice([i for i in indices if i != anchor_idx])
                negative_class = np.random.choice([c for c in self.class_indices if c != class_id])
                negative_idx = np.random.choice(self.class_indices[negative_class])

                d_ap = torch.dist(embeddings[anchor_idx], embeddings[positive_idx])
                d_an = torch.dist(embeddings[anchor_idx], embeddings[negative_idx])
                difficulty = (d_an - d_ap) / max_dist  # normalized 0..1

                if difficulty <= self.difficulty_threshold:
                    triplets.append((anchor_idx, positive_idx, negative_idx))
                if len(triplets) >= self.num_triplets_per_class:
                    break
        return triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a, p, n = self.triplets[idx]
        return self.embeddings[a], self.embeddings[p], self.embeddings[n]
    
# Define the embedding model (simple linear layer for fine-tuning)
class EmbeddingModel(nn.Module):
    def __init__(self, input_dim=768, embedding_dim=128):
        super().__init__()
        self.fc = nn.Linear(input_dim, embedding_dim)

    def forward(self, x):
        return self.fc(x)

# Initialize model, loss, optimizer
embedding_dim = 128
model = EmbeddingModel(input_dim=X_train.shape[1], embedding_dim=embedding_dim).to(device)
criterion = nn.TripletMarginLoss(margin=margin)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    # Gradual threshold: start at 0.2 easiest, go to 0.8 hardest
    difficulty_threshold = 0.2 + (0.8 * (epoch / (num_epochs - 1)))

    dataset = TripletDataset(X_train, y_train, num_triplets_per_class=50,
                             difficulty_threshold=difficulty_threshold)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    model.train()
    for anchor, positive, negative in dataloader:
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
        optimizer.zero_grad()
        anchor_emb = model(anchor)
        pos_emb = model(positive)
        neg_emb = model(negative)
        loss = criterion(anchor_emb, pos_emb, neg_emb)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, threshold {difficulty_threshold:.2f}, Loss: {loss.item()}")

# Evaluation on test set
model.eval()
with torch.no_grad():
    test_embeddings = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

# Compute cosine similarity and recall@k
def recall_at_k(query_emb, gallery_emb, query_labels, gallery_labels, k=5):
    similarities = cosine_similarity(query_emb, gallery_emb)
    recalls = []
    for i, sim in enumerate(similarities):
        sorted_indices = np.argsort(sim)[::-1]
        top_k_indices = sorted_indices[1:k+1]  # Exclude self
        top_k_labels = gallery_labels[top_k_indices]
        if query_labels[i] in top_k_labels:
            recalls.append(1)
        else:
            recalls.append(0)
    return np.mean(recalls)

recall_1 = recall_at_k(test_embeddings, test_embeddings, y_test, y_test, k=1)
recall_5 = recall_at_k(test_embeddings, test_embeddings, y_test, y_test, k=5)
print(f"Recall@1: {recall_1}, Recall@5: {recall_5}")

Epoch 1, threshold 0.20, Loss: 0.6403432488441467
Epoch 2, threshold 0.24, Loss: 0.07572373747825623
Epoch 3, threshold 0.28, Loss: 0.2155645787715912
Epoch 4, threshold 0.33, Loss: 0.2281852513551712
Epoch 5, threshold 0.37, Loss: 0.11567529290914536
Epoch 6, threshold 0.41, Loss: 0.1115911602973938
Epoch 7, threshold 0.45, Loss: 0.1568746268749237
Epoch 8, threshold 0.49, Loss: 0.1859988123178482
Epoch 9, threshold 0.54, Loss: 0.11310150474309921
Epoch 10, threshold 0.58, Loss: 0.1585940569639206
Epoch 11, threshold 0.62, Loss: 0.1519160121679306
Epoch 12, threshold 0.66, Loss: 0.0985594242811203
Epoch 13, threshold 0.71, Loss: 0.17132869362831116
Epoch 14, threshold 0.75, Loss: 0.2277946025133133
Epoch 15, threshold 0.79, Loss: 0.062159061431884766
Epoch 16, threshold 0.83, Loss: 0.06243505701422691
Epoch 17, threshold 0.87, Loss: 0.17094501852989197
Epoch 18, threshold 0.92, Loss: 0.0
Epoch 19, threshold 0.96, Loss: 0.11755195260047913
Epoch 20, threshold 1.00, Loss: 0.054166845977

In [87]:
# Compute accuracy using KNN on the learned embeddings
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

train_embeddings = model(torch.tensor(X_train, dtype=torch.float32).to(device)).detach().cpu().numpy()
# test_embeddings = model(torch.tensor(X_test, dtype=torch.float32).to(device)).detach().cpu().numpy()

knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(train_embeddings, y_train)      # ✅ train on training set
y_pred = knn.predict(test_embeddings)   # ✅ predict on test set
accuracy = accuracy_score(y_test, y_pred)
print(f"Classification Accuracy: {accuracy}")

Classification Accuracy: 0.625


In [ ]:
result = {
    "megadescriptor_version": megadescriptor_version,
    "dataset_version": detection,
    "seed": seed,
    "query_ratio": query_ratio,
    "accuracy": accuracy,
    "loss_function": "TripletLoss"
}

result